# 08｜vit-pytorch 基础 ViT 源码对照阅读

我们已经独立完成 Tiny ViT，并在 CIFAR-10 上完成 300 个 epoch 的训练。当前最佳验证准确率约为 **89.74%**，测试准确率约为 **89.57%**。

这一课不再从头讲 ViT，而是把自己的实现和 `lucidrains/vit-pytorch` 中的基础 `ViT` 一一对应。重点不是背源码，而是确认：**别人换了一种写法以后，数据流仍然是不是同一个 ViT。**

参考：[`lucidrains/vit-pytorch`](https://github.com/lucidrains/vit-pytorch)，本地安装版本为 `1.17.7`。

## 1. 本课目标

完成后应能：

1. 在开源源码中定位 `ViT`、`Transformer`、`Attention` 和 `FeedForward`；
2. 把它们与自己的输入嵌入、注意力子层、MLP 子层和 Encoder 对应起来；
3. 跟踪两种实现的关键 shape；
4. 解释至少三处设计差异；
5. 修改 `pool` 配置并重新跑通一次前向传播。

In [1]:
from importlib.metadata import version
import inspect

import torch
import vit_pytorch.vit as open_source_module
from vit_pytorch import ViT as OpenSourceViT

from vit import TinyViT

print("torch 版本：", torch.__version__)
print("vit-pytorch 版本：", version("vit-pytorch"))
print("基础 ViT 源码位置：", inspect.getsourcefile(open_source_module.ViT))

torch 版本： 2.10.0
vit-pytorch 版本： 1.17.7
基础 ViT 源码位置： D:\PythonWorkSpace\anaconda\envs\dl-study\Lib\site-packages\vit_pytorch\vit.py


## 2. 先建立模块对应关系

| 我们的实现 | vit-pytorch | 共同作用 |
|---|---|---|
| `PatchEmbedding` | `ViT.to_patch_embedding` | 图片切成 patch，并映射到 token 维度 |
| `ViTInputEmbedding` | `cls_token + pos_embedding + dropout` | 加入 CLS 和位置编码 |
| `AttentionSubLayer` | `Attention` + 外部残差 | Pre-LN 多头自注意力 |
| `MLPSubLayer` | `FeedForward` + 外部残差 | Pre-LN 前馈网络 |
| `EncoderBlock` | `Transformer.layers` 中的一对模块 | Attention + FFN |
| `TransformerEncoder` | `Transformer` | 堆叠多个 Block，并做最终 LayerNorm |
| `ClassificationHead` | `mlp_head` | 把全局特征映射到 10 类 logits |

类名不同不重要。阅读源码时最可靠的方法是追踪输入输出 shape 和残差连接位置。

## 3. 使用完全相同的模型规模

当前 Tiny ViT 配置：图片 `32 x 32`、patch `4 x 4`、token 维度 `256`、6 个 Encoder Block、4 个注意力头、MLP 隐藏维度 `1024`。

下面让开源实现使用相同配置。这样比较的是**实现差异**，而不是模型规模差异。

In [2]:
common_config = dict(
    image_size=32,
    patch_size=4,
    num_classes=10,
    dim=256,
    depth=6,
    heads=4,
    mlp_dim=1024,
    dropout=0.1,
)

open_source_vit = OpenSourceViT(
    **common_config,
    dim_head=64,
    emb_dropout=0.0,
    pool="cls",
)
our_vit = TinyViT()

images = torch.randn(2, 3, 32, 32)
open_source_logits = open_source_vit(images)
our_logits = our_vit(images)

print("开源实现输出：", tuple(open_source_logits.shape))
print("我们的实现输出：", tuple(our_logits.shape))
assert open_source_logits.shape == our_logits.shape == (2, 10)

开源实现输出： (2, 10)
我们的实现输出： (2, 10)


## 4. 用 forward hook 跟踪开源实现的 shape

`forward hook` 不修改模型，只在指定模块完成前向传播后记录输出。我们关心三个位置：

1. Patch Embedding 后：$B \times 64 \times 256$；
2. Transformer 后：$B \times 65 \times 256$；
3. 分类头后：$B \times 10$。

Patch Embedding 后还没有 CLS，因此是 64 个 token；进入 Transformer 前已经加入 CLS，因此变成 65 个。

In [4]:
captured_shapes = {}

def save_shape(name):
    """返回一个 hook，把模块输出 shape 保存到 captured_shapes。"""
    def hook(module, inputs, output):
        captured_shapes[name] = tuple(output.shape)
    return hook

hooks = [
    open_source_vit.to_patch_embedding.register_forward_hook(
        save_shape("Patch Embedding")
    ),
    open_source_vit.transformer.register_forward_hook(
        save_shape("Transformer")
    ),
    open_source_vit.mlp_head.register_forward_hook(
        save_shape("Classification Head")
    ),
]

with torch.inference_mode():
    _ = open_source_vit(images)

for hook in hooks:
    hook.remove()

for name, shape in captured_shapes.items():
    print(f"{name:20s} -> {shape}")

assert captured_shapes["Patch Embedding"] == (2, 64, 256)
assert captured_shapes["Transformer"] == (2, 65, 256)
assert captured_shapes["Classification Head"] == (2, 10)

Patch Embedding      -> (2, 64, 256)
Transformer          -> (2, 65, 256)
Classification Head  -> (2, 10)


## 5. 不从第一行硬读：先定位四个入口

阅读顺序建议从外向内：

`ViT.forward → Transformer.forward → Attention.forward / FeedForward.forward`

先看 `ViT.forward` 可以掌握总数据流，再进入 Attention 的 Q、K、V 细节。

In [4]:
for source_class in (
    open_source_module.ViT,
    open_source_module.Transformer,
    open_source_module.Attention,
    open_source_module.FeedForward,
):
    _, start_line = inspect.getsourcelines(source_class)
    print(f"{source_class.__name__:12s} 从源码第 {start_line} 行开始")

print("\nAttention.forward 的实际源码：")
print(inspect.getsource(open_source_module.Attention.forward))

ViT          从源码第 85 行开始
Transformer  从源码第 66 行开始
Attention    从源码第 30 行开始
FeedForward  从源码第 15 行开始

Attention.forward 的实际源码：
    def forward(self, x):
        x = self.norm(x)

        qkv = self.to_qkv(x).chunk(3, dim = -1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        attn = self.attend(dots)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)



## 6. Attention 源码逐步对应

开源实现没有使用 `nn.MultiheadAttention`，而是手动完成：

1. `LayerNorm`：对应我们的 `layer_norm1`；
2. `to_qkv(x).chunk(3)`：一次线性映射后切成 Q、K、V；
3. `rearrange`：把特征拆成多个 head；
4. `q @ k.transpose(...) * scale`：计算注意力分数；
5. `Softmax`：把分数转换为注意力权重；
6. `attention @ v`：根据权重汇总 Value；
7. 合并多个 head，再做输出投影。

我们的 `nn.MultiheadAttention` 把第 2～7 步封装起来了。两种实现的数学流程相同，但开源版本更容易直接观察 Q、K、V 和 Attention Map。

## 7. 至少六处设计差异

### 差异一：Patch Embedding

- 我们：`Conv2d(kernel_size=4, stride=4)`；
- 开源：`Rearrange + LayerNorm(48) + Linear(48, 256) + LayerNorm(256)`。

不重叠卷积与“展平 patch 后接 Linear”在投影意义上等价，但开源实现额外加入了两个 LayerNorm。

### 差异二：QKV bias

- 我们的 `nn.MultiheadAttention` 默认带 QKV bias；
- 开源的 `to_qkv` 明确设置 `bias=False`。

### 差异三：`dim_head` 更灵活

开源实现允许 `heads x dim_head` 与 `dim` 不同；我们的实现要求 `embed_dim` 能被 `num_heads` 整除，内部总维度固定为 `embed_dim`。

### 差异四：残差连接放置位置

- 我们把残差写在 `AttentionSubLayer` 和 `MLPSubLayer` 内；
- 开源实现让 `Attention`、`FeedForward` 只返回分支结果，在 `Transformer.forward` 中统一执行 `+ x`。

### 差异五：池化方式

我们固定取 `x[:, 0]`；开源实现还支持 `pool='mean'`，即对所有 token 做平均。

### 差异六：输入尺寸灵活性

开源 `pair()` 支持高宽元组，并会按实际序列长度截取位置编码；我们的第一版固定使用 `32 x 32` 方形输入，错误会更早暴露。

In [5]:
our_parameters = sum(p.numel() for p in our_vit.parameters())
open_source_parameters = sum(p.numel() for p in open_source_vit.parameters())

print(f"我们的参数量：{our_parameters:,}")
print(f"开源参数量：  {open_source_parameters:,}")
print(f"相差参数量：  {our_parameters - open_source_parameters:,}")

# 差异主要来自：我们的 6 层 QKV bias，多出 6 x 3 x 256 = 4608 个参数；
# 开源 Patch Embedding 的两个 LayerNorm 又多出 608 个参数。
# 因此净差值为 4608 - 608 = 4000。
assert our_parameters - open_source_parameters == 4000

我们的参数量：4,771,082
开源参数量：  4,767,082
相差参数量：  4,000


## 8. 小实验：把 CLS pooling 改成 mean pooling

`pool='cls'` 使用经过 Encoder 更新后的 CLS token：

$$x_{global} = x[:, 0]$$

`pool='mean'` 则对所有 patch token 的输出求平均，不再创建 CLS token：

$$x_{global} = \frac{1}{N}\sum_{i=1}^{N}x_i$$

这不是说 mean 一定更好，而是说明“如何得到全局图片表示”本身也是一个可以实验的设计选择。

In [6]:
mean_pool_vit = OpenSourceViT(
    **common_config,
    dim_head=64,
    emb_dropout=0.0,
    pool="mean",
)

with torch.inference_mode():
    mean_pool_logits = mean_pool_vit(images)

print("mean pooling 输出：", tuple(mean_pool_logits.shape))
print("CLS token 参数 shape：", tuple(mean_pool_vit.cls_token.shape))
print("位置编码 shape：", tuple(mean_pool_vit.pos_embedding.shape))

assert mean_pool_logits.shape == (2, 10)
assert mean_pool_vit.cls_token.shape[0] == 0

mean pooling 输出： (2, 10)
CLS token 参数 shape： (0, 256)
位置编码 shape： (64, 256)


## 9. 为什么不能直接互换权重

即使两个模型的输入输出 shape 相同，也不能直接把我们的 checkpoint 加载到 `vit-pytorch`：

- 参数名称不同；
- Patch Embedding 结构不同；
- QKV bias 配置不同；
- Q、K、V 权重的存储组织方式不同。

如果真的要迁移，需要逐项建立参数映射，而不是直接调用 `load_state_dict()`。这一点说明：**结构等价不等于参数文件兼容。**

## 10. 本课总结

我们已经完成学习计划阶段四的第一遍基础源码对照：

- 从 `ViT.forward` 找到完整数据流；
- 从 `Transformer.forward` 找到两次残差相加；
- 从 `Attention.forward` 看见手写 Q、K、V 与 Softmax；
- 将开源模块与自己的模块逐一对应；
- 解释了六处实现差异；
- 把 CLS pooling 改为 mean pooling 并重新跑通。

下一步可以继续做更有 ViT 特征的实战：**提取 Attention Map，观察 CLS 最关注图片的哪些 patch。**

## 11. 自测问题

1. 为什么 `to_patch_embedding` 输出 64 个 token，而 `Transformer` 输出 65 个？
2. `to_qkv(x).chunk(3, dim=-1)` 在做什么？
3. 开源实现的残差连接为什么不写在 `Attention.forward` 内？
4. 为什么两种 Patch Embedding 写法都能得到 `B x 64 x 256`？
5. `pool='mean'` 时为什么 CLS token 的数量为 0？
6. 两个模型输出 shape 相同，为什么 checkpoint 仍然不能直接互换？